In [2]:
import pandas as pd
df=pd.read_csv('/content/IMDB Dataset.csv',encoding='latin1')
print(df.head())
print(df['sentiment'].value_counts())

                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive
sentiment
positive    25000
negative    25000
Name: count, dtype: int64


In [3]:
df['sentiment']=df['sentiment'].map({
    'positive':1,
    'negative':0
})

In [10]:
import re

negation_words=[
    "not good","not bad","not great","don't like","didn't like","never liked","wasn't good","isn't good","no good"
]
def clean_text(text):
  text=text.lower()
  text=re.sub(r"[^a-zA-Z\s']"," ",text)
  for phrase in negation_words:
    text=text.replace(phrase,phrase.replace(" ","_"))
  return text

In [ ]:
df['review']=df['review'].apply(clean_text)

In [13]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(df['review'], df['sentiment'], test_size=0.2, random_state=42)

In [16]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
vocab_size=20000
max_len=250
tokenizer=Tokenizer(num_words=vocab_size,oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)
X_train_seq=tokenizer.texts_to_sequences(X_train)
X_test_seq=tokenizer.texts_to_sequences(X_test)
x_train_pad=pad_sequences(X_train_seq,maxlen=max_len,padding='post')
x_test_pad=pad_sequences(X_test_seq,maxlen=max_len,padding='post')

In [18]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
model=Sequential([
    Embedding(vocab_size,128,input_length=max_len),
    LSTM(128,dropout=0.3,recurrent_dropout=0.3),
    Dense(64,activation='relu'),
    Dropout(0.3),
    Dense(1,activation='sigmoid')
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [19]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history=model.fit(
    x_train_pad,
    y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.2
)

Epoch 1/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 450s 888ms/step - accuracy: 0.5364 - loss: 0.6760 - val_accuracy: 0.5745 - val_loss: 0.6954
Epoch 2/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 433s 865ms/step - accuracy: 0.6044 - loss: 0.6115 - val_accuracy: 0.5844 - val_loss: 0.6243
Epoch 3/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 444s 871ms/step - accuracy: 0.6495 - loss: 0.5951 - val_accuracy: 0.6370 - val_loss: 0.6161
Epoch 4/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 427s 854ms/step - accuracy: 0.7597 - loss: 0.4692 - val_accuracy: 0.8635 - val_loss: 0.3449
Epoch 5/5
423/500 ━━━━━━━━━━━━━━━━━━━━ 1:03 823ms/step - accuracy: 0.9043 - loss: 0.2587

In [23]:
loss, acc=model.evaluate(x_test_pad, y_test)
print("Test Accuracy:",acc)

313/313 ━━━━━━━━━━━━━━━━━━━━ 35s 110ms/step - accuracy: 0.8849 - loss: 0.2871
Test Accuracy: 0.8848999738693237


In [27]:
def predict_sentiment(review):
  revview=clean_text(review)
  seq=tokenizer.texts_to_sequences([review])
  padded=pad_sequences(seq,maxlen=max_len,padding='post')
  prediction=model.predict(padded)[0][0]
  print("\nReview",review)
  print("Score:",prediction)
  if prediction>=0.5:
    print("Sentiment:Positive")
  else:
    print("Sentiment:Negative")

In [28]:
predict_sentiment("I hate this game.")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 85ms/step

Review I hate this game.
Score: 0.39298922
Sentiment:Negative
